# Mapa IPM por dimension - Barrio Obrero

Coropletas por dimension del IPM (5 dimensiones DANE: Educacion, Ninez y juventud,
Trabajo, Salud, Vivienda y servicios) a nivel de manzana, conmutables. Al hacer click en
una manzana, el panel dibuja barras con sus privaciones. Unidad: % de personas
(Censo 2018, DANE). Vacios = 0.


## 0. Setup Colab (opcional)

In [ ]:
# === SETUP COLAB (se ignora en local) ===
def _en_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False
if _en_colab():
    import os,subprocess,sys
    REPO='https://github.com/j0rg3c45/mapa_Analitico_ciudad_Paraiso.git'
    DEST='/content/mapa_Analitico_ciudad_Paraiso'
    subprocess.run([sys.executable,'-m','pip','install','-q','geopandas','folium','mapclassify','pyarrow','openpyxl'],check=False)
    if not os.path.isdir(DEST): subprocess.run(['git','clone','--depth','1',REPO,DEST],check=False)
    if os.path.isdir(os.path.join(DEST,'notebooks')): os.chdir(os.path.join(DEST,'notebooks'))
    print('Colab listo:',os.getcwd())
else:
    print('Entorno local (uv). Setup Colab omitido.')


## 1. Entorno

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
OKI_AZUL="#0072B2"; OKI_GRIS="#333333"; BG="#F4F6F9"; CRS_WEB=4326
print("Entorno listo.")

## 2. Rutas, poligono y bounds

In [ ]:
def _ruta_bo(nombre):
    for p in [os.path.join("..","data","Barrio_Obrero_geojson",nombre),
              os.path.join("data","Barrio_Obrero_geojson",nombre),
              os.path.join("/content","data","Barrio_Obrero_geojson",nombre)]:
        if os.path.isfile(p): return os.path.abspath(p)
    return None
POLIGONO_ZONA=gpd.read_file(_ruta_bo("poligono_Barrio_Obrero_V1.geojson")).to_crs(CRS_WEB)
_b=POLIGONO_ZONA.total_bounds
BO_BOUNDS=[[float(_b[1]),float(_b[0])],[float(_b[3]),float(_b[2])]]
OUT_DIR=None
for _d in [os.path.join("..","data_processed_geo"),"data_processed_geo","/content/data_processed_geo"]:
    try:
        os.makedirs(_d,exist_ok=True); OUT_DIR=os.path.abspath(_d); break
    except Exception: continue
MAPAS_DIR=os.path.abspath(os.path.join(os.path.dirname(OUT_DIR.rstrip(os.sep)),"mapa_html","Barrio_obrero"))
os.makedirs(MAPAS_DIR,exist_ok=True)
print("Salida:", MAPAS_DIR)

## 3. Carga del IPM (15 variables por manzana)

In [ ]:
_rp=_ruta_bo(os.path.join("IPM_BARRIO_OBRERO_2_POLIGONOS","IPM_Barrio_Obrero_V1.geojson"))
ipm_z=gpd.read_file(_rp).to_crs(CRS_WEB)
ipm_z=ipm_z[ipm_z.geometry.notna()].copy()
ipm_z["ipm"]=pd.to_numeric(ipm_z["ipm"], errors="coerce")
_z=POLIGONO_ZONA.geometry.union_all()
ipm_z=ipm_z[ipm_z.geometry.within(_z) | ipm_z.geometry.intersects(_z)].reset_index(drop=True)
print("Manzanas:", len(ipm_z))

## 4. Exportacion de lamina (deck) - helper reutilizable

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium UN boton de exportacion
# de lamina con estructura de deck (PMU). Captura el contenedor del mapa como
# imagen y la compone dentro de una lamina HTML (encabezado + 2 columnas +
# pie) que se exporta a PDF (una pagina). Este archivo es solo una plantilla
# de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar el mapa se oculta el panel de tiles base (.leaflet-tile-pane)
# y se pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales (contorno, manzanas, puntos, calor). La imagen del mapa
# va en la columna derecha de la lamina.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con UN boton:
    'Exportar lamina (deck)'. Al pulsar, captura el contenedor del mapa a una
    imagen y construye una lamina HTML con estructura de deck del PMU
    (encabezado dorado, 2 columnas: resumen/cifras clonadas del panel +
    mapa, pie con corte y fuente); captura la lamina y la exporta a PDF de
    una pagina (landscape) con dialogo "Guardar como" (showSaveFilePicker) y
    fallback a descarga. Nombre: deck_<slug>_<AAAA-MM-DD>.pdf. Debe llamarse
    ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo.
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl___MID__" style="position:fixed;top:12px;right:12px;z-index:10000;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btndeck___MID__" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar lamina (deck)</button>
</div>
<script>
(function(){
  var MAP_ID = '__MID__';
  var TITULO = '__TITULO__';
  var SLUG = '__SLUG__';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';
  var DORADO = '#8A6D1A';
  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // Paso (a): captura SOLO el contenedor del mapa a un canvas, ocultando antes
  // los botones de export, el selector de capas, los tiles base (CORS), el
  // panel flotante de graficas y el slider de intensidad del calor; restaura
  // todo despues. Devuelve el canvas via callback.
  function capturarMapa(cont, cb){
    var ctl = document.getElementById('exportctl_'+MAP_ID);
    if(ctl) ctl.style.display = 'none';
    var layerCtls = cont.querySelectorAll('.leaflet-control-layers');
    layerCtls.forEach(function(lc){ lc.dataset.prevDisplay = lc.style.display; lc.style.display = 'none'; });
    // Panel de graficas flotante (#seg-panel o #eu-panel) y slider del calor.
    var panel = cont.querySelector('#seg-panel, #eu-panel, #ipm-panel');
    var prevPanelDisplay = null;
    if(panel){ prevPanelDisplay = panel.style.display; panel.style.display = 'none'; }
    var slider = cont.querySelector('#heat-intensity-ctrl');
    var prevSliderDisplay = null;
    if(slider){ prevSliderDisplay = slider.style.display; slider.style.display = 'none'; }
    // CORS: ocultar tiles base y aplicar fondo claro.
    var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
    var fondoPrev = cont.style.background;
    cont.style.background = '#F4F6F9';
    tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
    function restaurar(){
      if(ctl) ctl.style.display = '';
      layerCtls.forEach(function(lc){ lc.style.display = ''; });
      if(panel) panel.style.display = (prevPanelDisplay || '');
      if(slider) slider.style.display = (prevSliderDisplay || '');
      tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
      cont.style.background = fondoPrev;
    }
    html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
      .then(function(canvas){ restaurar(); cb(null, canvas); })
      .catch(function(err){ restaurar(); cb(err, null); });
  }
  // Paso previo: asegura el panel #seg-panel/#eu-panel expandido (si usa
  // <details>) para que Chart.js tenga el canvas visible y pintado, luego
  // captura el canvas real del grafico a un dataURL PNG. Devuelve null si
  // no hay canvas o esta vacio. cb(dataUrl|null).
  function capturarGrafico(cb){
    var panel = document.querySelector('#seg-panel, #eu-panel, #ipm-panel');
    if(!panel){ cb(null); return; }
    var dets = panel.querySelectorAll('details');
    dets.forEach(function(d){ d.setAttribute('open', 'open'); });
    // Pequeno tiempo para que el canvas quede visible y Chart.js pinte.
    setTimeout(function(){
      var cv = panel.querySelector('canvas[id*="chart"]') || panel.querySelector('canvas');
      if(!cv || !cv.width || !cv.height){ cb(null); return; }
      try { cb(cv.toDataURL('image/png')); }
      catch(e){ cb(null); }
    }, 250);
  }
  // Paso (b): construye el div-lamina temporal con estructura de deck.
  // La imagen del mapa (canvas de (a)) va en la columna derecha; el panel
  // existente (si lo hay) se clona en la columna izquierda.
  function construirLamina(mapaDataUrl, graficoDataUrl){
    var lam = document.createElement('div');
    lam.id = 'deck-lamina';
    lam.style.cssText = 'position:fixed;top:0;left:0;z-index:2147483647;'+
      'width:1200px;background:#FFFFFF;color:#333;'+
      'font-family:system-ui,Segoe UI,sans-serif;'+
      'box-sizing:border-box;overflow:hidden;';
    // ENCABEZADO (barra con titulo + linea de seccion gris + borde dorado).
    var head = document.createElement('div');
    head.style.cssText = 'padding:16px 22px 12px 22px;border-bottom:3px solid '+DORADO+';';
    head.innerHTML =
      '<div style="font-size:20px;font-weight:700;color:#222;line-height:1.2">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#888;margin-top:4px">ITT Ciudad Paraiso - Lamina de deck (PMU)</div>';
    lam.appendChild(head);
    // CUERPO en 2 columnas (grid).
    var body = document.createElement('div');
    body.style.cssText = 'display:grid;grid-template-columns:35% 65%;gap:16px;'+
      'padding:16px 22px;align-items:start;';
    // Columna izquierda: resumen / cifras (clon del panel si existe).
    var colIzq = document.createElement('div');
    colIzq.style.cssText = 'min-width:0;';
    var panelOrig = document.querySelector('#seg-panel, #eu-panel, #ipm-panel');
    if(panelOrig){
      var clon = panelOrig.cloneNode(true);
      clon.removeAttribute('id');
      // Sacar el clon del flujo fijo para que fluya dentro de la columna.
      clon.style.position = 'static';
      clon.style.top = 'auto'; clon.style.left = 'auto';
      clon.style.right = 'auto'; clon.style.bottom = 'auto';
      clon.style.zIndex = 'auto';
      clon.style.maxWidth = 'none';
      clon.style.width = '100%';
      clon.style.boxShadow = 'none';
      // Expandir cualquier <details> para que la serie se vea.
      var dets = clon.querySelectorAll('details');
      dets.forEach(function(d){ d.setAttribute('open', 'open'); });
      // El canvas de Chart.js se clona VACIO (los canvas no conservan su
      // contenido al clonar). Lo sustituimos por un <img> con la imagen
      // capturada del canvas real para que el grafico SE VEA en la lamina.
      if(graficoDataUrl){
        var cvClon = clon.querySelector('canvas');
        if(cvClon){
          var imgChart = document.createElement('img');
          imgChart.src = graficoDataUrl;
          imgChart.style.cssText = 'display:block;width:100%;height:auto;';
          if(cvClon.parentNode){ cvClon.parentNode.replaceChild(imgChart, cvClon); }
        }
      }
      colIzq.appendChild(clon);
    } else {
      colIzq.innerHTML =
        '<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px">'+TITULO+'</div>'+
        '<div style="font-size:13px;color:#666">Resumen no disponible</div>';
    }
    // Columna derecha: imagen del mapa escalada al ancho de la columna.
    var colDer = document.createElement('div');
    colDer.style.cssText = 'min-width:0;';
    var img = document.createElement('img');
    img.src = mapaDataUrl;
    img.style.cssText = 'display:block;width:100%;height:auto;'+
      'border:1px solid #d9dde3;border-radius:6px;';
    colDer.appendChild(img);
    body.appendChild(colIzq);
    body.appendChild(colDer);
    lam.appendChild(body);
    // PIE (corte + fuente + nota).
    var pie = document.createElement('div');
    pie.style.cssText = 'padding:10px 22px 14px 22px;border-top:1px solid #e2e5ea;'+
      'font-size:11px;color:#777;';
    pie.innerHTML = 'Corte: '+new Date().toLocaleString()+
      '  |  Fuente: '+FUENTE+'  |  Mapa ilustrativo';
    lam.appendChild(pie);
    document.body.appendChild(lam);
    return lam;
  }
  // Dialogo "Guardar como" (Chrome/Edge) con fallback a descarga clasica.
  function guardarPDF(blob, nombre){
    function descargaFallback(){
      var url = URL.createObjectURL(blob);
      var a = document.createElement('a');
      a.href = url; a.download = nombre;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      setTimeout(function(){ URL.revokeObjectURL(url); }, 1500);
    }
    if(window.showSaveFilePicker){
      var tipos = [{description:'Documento PDF', accept:{'application/pdf':['.pdf']}}];
      window.showSaveFilePicker({suggestedName:nombre, types:tipos})
        .then(function(handle){ return handle.createWritable(); })
        .then(function(writable){ return writable.write(blob).then(function(){ return writable.close(); }); })
        .catch(function(e){
          // Si el usuario cancela (AbortError) no se descarga nada; otros -> fallback.
          if(e && e.name === 'AbortError'){ return; }
          descargaFallback();
        });
    } else {
      descargaFallback();
    }
  }
  // Bounds del poligono de la zona de estudio (Ciudad Paraiso) para
  // reencuadrar el mapa SOLO en la generacion de la lamina (no afecta la
  // vista interactiva previa del usuario; tras exportar queda en este
  // encuadre pero la interactividad se mantiene).
  var ZONA_BOUNDS = L.latLngBounds([[3.442445,-76.53259],[3.44791,-76.526004]]);
  function exportar(){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    // 1) Forzar el panel a 'open' y capturar el grafico real a dataURL.
    capturarGrafico(function(graficoDataUrl){
      var mp = window[MAP_ID];
      // 2) fitBounds al poligono. maxZoom ENTERO evita el zoom fraccional que
      //    descoloca las capas en html2canvas (desfase al imprimir).
      var capturado = false;
      function trasEncuadre(){
        if(capturado){ return; }
        capturado = true;
        try { if(mp){ mp.invalidateSize(false); } } catch(e){}
        // 3) Margen corto tras asentar panes/tiles, luego capturar.
        setTimeout(function(){
        // 4) Capturar el mapa (ocultando botones/tiles/panel/slider).
        capturarMapa(cont, function(err, canvasMapa){
          if(err){ alert('Error al capturar el mapa: '+err); return; }
          var mapaDataUrl = canvasMapa.toDataURL('image/jpeg', 0.95);
          // 5) Construir lamina (cifras + grafico como imagen + mapa).
          var lam = construirLamina(mapaDataUrl, graficoDataUrl);
          function limpiar(){
            if(lam && lam.parentNode) lam.parentNode.removeChild(lam);
          }
          // 6) Capturar la lamina y generar el PDF.
          html2canvas(lam, {backgroundColor:'#FFFFFF', logging:false, scale:2, useCORS:true})
            .then(function(canvas){
              limpiar();
              var imgData = canvas.toDataURL('image/jpeg', 0.95);
              var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
              var w = canvas.width, hh = canvas.height;
              var orient = (w >= hh) ? 'landscape' : 'portrait';
              var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
              doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
              var blob = doc.output('blob');
              var nombre = 'deck_'+SLUG+'_'+fechaISO()+'.pdf';
              guardarPDF(blob, nombre);
            })
            .catch(function(e){
              limpiar();
              alert('Error al exportar la lamina: '+e);
            });
        });
        }, 350);
      }
      // Disparar fitBounds y esperar 'moveend'; fallback por timeout si no llega.
      try {
        if(mp){
          mp.once('moveend', trasEncuadre);
          mp.fitBounds(ZONA_BOUNDS, {padding:[10,10], maxZoom: 17, animate: false});
          setTimeout(trasEncuadre, 1200);
        } else {
          trasEncuadre();
        }
      } catch(e){ trasEncuadre(); }
    });
  }
  function init(){
    var b = document.getElementById('btndeck_'+MAP_ID);
    if(b) b.addEventListener('click', function(){ exportar(); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
"""
    bloque = (bloque.replace("__MID__", map_id)
                    .replace("__TITULO__", titulo_js)
                    .replace("__SLUG__", slug_js))
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa

## 5. Construccion del mapa (Folium)

In [ ]:
# 4a. Mapa IPM por DIMENSION - Barrio Obrero. Coropletas conmutables + datos por manzana.
import folium
DIM = {
    "Educacion": ["analf_","bajo_","infancia_","inasis_","rezago_"],
    "Ninez y juventud": ["infancia_","inasis_","rezago_","trab_infan_"],
    "Trabajo": ["trab_infan_","depen_","infor_"],
    "Salud": ["salud_","asegu_"],
    "Vivienda y servicios": ["haci_","pared_","pisos_","agua_","excre_"],
}
VAR_LABEL={"analf_":"Analfabetismo","bajo_":"Bajo logro educativo","infancia_":"Barreras primera infancia",
  "inasis_":"Inasistencia escolar","rezago_":"Rezago escolar","trab_infan_":"Trabajo infantil",
  "depen_":"Dependencia economica","infor_":"Trabajo informal","salud_":"Barreras de salud",
  "asegu_":"Sin aseguramiento","haci_":"Hacinamiento critico","pared_":"Paredes precarias",
  "excre_":"Sin alcantarillado","pisos_":"Pisos precarios","agua_":"Sin acceso a agua"}
VARS=list(VAR_LABEL.keys())
# Normalizar columnas a minusculas y rellenar vacios = 0 (REDATAM).
import pandas as _pd
for v in VARS:
    if v.upper() in ipm_z.columns:
        _serie = ipm_z[v.upper()]
    elif v in ipm_z.columns:
        _serie = ipm_z[v]
    else:
        _serie = _pd.Series([0.0] * len(ipm_z), index=ipm_z.index)
    ipm_z[v] = _pd.to_numeric(_serie, errors="coerce").fillna(0.0)
for _d,_cols in DIM.items():
    ipm_z["dim_"+_d] = ipm_z[_cols].mean(axis=1)
# Paleta daltonica (azul->naranja) continua 0-100.
def _col_calor(v):
    if v is None or (isinstance(v,float) and pd.isna(v)): return "#bdbdbd"
    v=max(0.0,min(100.0,float(v)))
    if v<=50:
        t=v/50.0; r=int(44+t*(166-44)); g=int(127+t*(189-127)); b=int(184+t*(219-184))  # azul->azul claro
    else:
        t=(v-50)/50.0; r=int(253+t*(227-253)); g=int(174+t*(74-174)); b=int(97+t*(51-97))  # naranja->rojo-naranja
    return "#%02x%02x%02x" % (r,g,b)
_c=POLIGONO_ZONA.geometry.union_all().centroid
m_dim=folium.Map(location=[_c.y,_c.x], zoom_start=17, tiles=None, control_scale=True,
                 zoom_snap=0.25, zoom_delta=0.1, max_zoom=22)
m_dim.get_root().width="100%"; m_dim.get_root().height="100%"
m_dim.get_root().header.add_child(folium.Element(
    "<style>html,body{height:100%;margin:0;padding:0}.folium-map{position:absolute;top:0;bottom:0;left:0;right:0;width:100%;height:100%}</style>"))
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",attr="Google",name="Google Streets",show=True,max_zoom=22,max_native_zoom=20).add_to(m_dim)
folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",attr="Esri",name="Esri Satelite",show=False,max_zoom=22,max_native_zoom=20).add_to(m_dim)
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",attr="Google",name="Google Satelite",show=False,max_zoom=22,max_native_zoom=20).add_to(m_dim)
_zg=POLIGONO_ZONA.geometry.union_all(); _mundo=[[-180,-85],[180,-85],[180,85],[-180,85],[-180,-85]]; _ag=[]
for _gm in (list(_zg.geoms) if _zg.geom_type=="MultiPolygon" else [_zg]): _ag.append([[x,y] for x,y in _gm.exterior.coords])
folium.GeoJson({"type":"Feature","properties":{},"geometry":{"type":"Polygon","coordinates":[_mundo]+_ag}},name="Mascara (foco en la zona)",control=False,
    style_function=lambda f:{"fillColor":"#000","color":"#000","weight":0,"fillOpacity":0.2}).add_to(m_dim)
folium.GeoJson(POLIGONO_ZONA.to_json(), name="Limite Barrio Obrero", show=True,
    style_function=lambda f:{"color":"#1B4F8A","weight":5,"dashArray":"10 6","fill":False}).add_to(m_dim)
ipm_z=ipm_z.reset_index(drop=True); ipm_z["mz_id"]=range(len(ipm_z))
MZ_DIM=[]
for _,r in ipm_z.iterrows():
    MZ_DIM.append({"id":int(r["mz_id"]),"ipm":(round(float(r["ipm"]),1) if pd.notna(r.get("ipm")) else None),
        "dims":{d:round(float(r["dim_"+d]),1) for d in DIM},
        "vars":{VAR_LABEL[v]:round(float(r[v]),1) for v in VARS}})
_fg_dim={}
for _d in DIM:
    _fg=folium.FeatureGroup(name="Dim: "+_d, show=(_d=="Trabajo"))
    for _,r in ipm_z.iterrows():
        _v=r.get("dim_"+_d); _col=_col_calor(_v); _mid=int(r["mz_id"])
        _gj=folium.GeoJson(r["geometry"].__geo_interface__,
            style_function=(lambda f,c=_col:{"fillColor":c,"color":"#ffffff","weight":0.5,"fillOpacity":0.8}),
            highlight_function=lambda f:{"weight":2.5,"color":"#111","fillOpacity":0.9},
            tooltip="%s: %.1f%%" % (_d, float(_v)))
        _gj.add_child(folium.Popup("mz:"+str(_mid), show=False)); _gj.add_to(_fg)
    _fg_dim[_d]=_fg
    _fg.add_to(m_dim)
m_dim.fit_bounds(BO_BOUNDS, padding=(30,30))
folium.LayerControl(collapsed=False, position="topright").add_to(m_dim)
print("Mapa IPM dimension Barrio Obrero construido. Manzanas:", len(MZ_DIM))

## 6. Panel reactivo: barras por manzana al dar clic

In [ ]:
# 5a. Panel reactivo Barrio Obrero: barras por manzana al hacer click.
_MAP_ID=m_dim.get_name()
_prom_dim={d:round(float(ipm_z["dim_"+d].mean()),1) for d in DIM}
_prom_var={VAR_LABEL[v]:round(float(ipm_z[v].mean()),1) for v in VARS}
_PAYLOAD=json.dumps({"manzanas":MZ_DIM,"prom_dim":_prom_dim,"prom_var":_prom_var,"dims":list(DIM.keys())}, ensure_ascii=False)
_panel = """
<div id="ipm-panel" style="position:fixed;top:16px;left:16px;z-index:9999;
     background:__BG__;width:min(430px,43vw);max-height:92vh;overflow:auto;
     padding:12px 14px;border-radius:12px;box-shadow:0 2px 12px rgba(0,0,0,.3);
     font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="font-weight:700;font-size:14px;margin-bottom:2px">IPM por dimension - Barrio Obrero</div>
  <div id="dim-sub" style="font-size:11px;color:#666;margin-bottom:8px">Promedio de la zona. Toca una manzana para ver su detalle. Unidad: % de personas (Censo 2018).</div>
  <div style="font-weight:700;font-size:12px;margin:4px 0">Dimensiones (%)</div>
  <div style="position:relative;height:150px;width:100%"><canvas id="dim-chart"></canvas></div>
  <div style="font-weight:700;font-size:12px;margin:10px 0 4px">Variables (%)</div>
  <div style="position:relative;height:320px;width:100%"><canvas id="var-chart"></canvas></div>
</div>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<script>
(function(){
  var P=__PAYLOAD__; var MAP="__MAP_ID__"; var dimCh=null,varCh=null;
  function colorDim(v){ if(v==null)return "#bdbdbd"; v=Math.max(0,Math.min(100,v));
    if(v<=50){var t=v/50;return 'rgb('+Math.round(44+t*122)+','+Math.round(127+t*62)+','+Math.round(184+t*35)+')';}
    var t=(v-50)/50;return 'rgb('+Math.round(253-t*26)+','+Math.round(174-t*100)+','+Math.round(97-t*46)+')';}
  function barras(id,labels,data,prev){
    var cv=document.getElementById(id); if(!cv||typeof Chart==='undefined')return null;
    if(prev)prev.destroy();
    return new Chart(cv.getContext('2d'),{type:'bar',data:{labels:labels,datasets:[{data:data,backgroundColor:data.map(colorDim),borderColor:'#555',borderWidth:0.5}]},
      options:{indexAxis:'y',responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{callbacks:{label:function(c){return c.parsed.x+'%';}}}},
        scales:{x:{beginAtZero:true,max:100,ticks:{font:{size:9}}},y:{ticks:{font:{size:9}}}}}});
  }
  function pintar(dimObj,varObj,titulo){
    document.getElementById('dim-sub').textContent=titulo;
    dimCh=barras('dim-chart',P.dims,P.dims.map(function(d){return dimObj[d];}),dimCh);
    var it=Object.keys(varObj).map(function(k){return [k,varObj[k]];}).sort(function(a,b){return b[1]-a[1];});
    varCh=barras('var-chart',it.map(function(x){return x[0];}),it.map(function(x){return x[1];}),varCh);
  }
  function vista(){ pintar(P.prom_dim,P.prom_var,'Promedio de la zona. Toca una manzana para ver su detalle.'); }
  function sel(id){ var mz=P.manzanas.find(function(m){return m.id===id;}); if(!mz)return;
    pintar(mz.dims,mz.vars,'Manzana seleccionada'+(mz.ipm!=null?(' - IPM '+mz.ipm+'%'):'')+' (toca otra para cambiar)'); }
  function hook(){ var map=window[MAP]; if(!map){setTimeout(hook,300);return;}
    map.on('popupopen',function(e){ try{var t=e.popup.getContent();
      if(typeof t==='string'&&t.indexOf('mz:')===0){ if(e.popup.remove)e.popup.remove(); sel(parseInt(t.split(':')[1],10)); }}catch(err){} }); }
  function init(){ vista(); hook(); }
  if(document.readyState==='complete'){setTimeout(init,500);}else{window.addEventListener('load',function(){setTimeout(init,500);});}
})();
</script>
"""
_panel=(_panel.replace("__BG__",BG).replace("__GRIS__",OKI_GRIS).replace("__PAYLOAD__",_PAYLOAD).replace("__MAP_ID__",_MAP_ID))
m_dim.get_root().html.add_child(folium.Element(_panel))
print("Panel reactivo dimension Barrio Obrero inyectado.")

## 7. Exportacion del HTML

In [ ]:
agregar_export(m_dim, "IPM por dimension - Barrio Obrero", "bo_ipm_dimensiones")
html=os.path.join(MAPAS_DIR,"bo_mapa_ipm_dimensiones.html")
m_dim.save(html)
print("Mapa exportado:", html)
m_dim